# Experiment 01: Classical 4-bit XOR Cipher & Known-Plaintext Attack (KPA)

> **Educational & Pedagogical Reference Baseline**
>
> This notebook establishes the **classical cryptographic baseline** for the toy XOR cipher before transitioning to Grover's Quantum Search Algorithm in Part 2.
>
> Here we cover:
> 1. **Cryptographic Model & Parameters**: 4-bit block and key size ($N = 2^4 = 16$).
> 2. **Classical Encryption**: $C = P \oplus K$.
> 3. **Classical Decryption**: $P = C \oplus K$.
> 4. **Known-Plaintext Attack (KPA) Threat Model**: Given pair $(P, C)$, recover $K$.
> 5. **Classical Search Attack (Brute-Force)**: Exhaustive search over keyspace $\mathcal{O}(N)$.
> 6. **Direct Algebraic Inversion**: $K = P \oplus C$ (specific to XOR).
> 7. **Comparison with Grover's Quantum Search**: Preparing for $\mathcal{O}(\sqrt{N})$.


## 1. Imports & Helper Functions
We define binary representation helpers to trace bitwise values clearly.

In [1]:
import numpy as np

def to_bin(val: int, width: int = 4) -> str:
    """Converts an integer to a zero-padded binary string of specified width."""
    return f"{val:0{width}b}"

def print_bits(label: str, val: int, width: int = 4):
    """Pretty prints a label, binary value, and decimal representation."""
    print(f"{label:<22}: {to_bin(val, width)} (decimal: {val})")

print("Helpers initialized successfully.")

Helpers initialized successfully.


## 2. Define Experiment Parameters

We use the exact parameters shared with the quantum implementation in **Notebook 02**:
- **Block Size**: 4 bits
- **Key Size**: 4 bits
- **Search Space ($N$)**: $2^4 = 16$ candidate keys ($k \in [0, 15]$)
- **Known Plaintext ($P$)**: `1010` ($10_{10}$)
- **Secret Key ($K$)**: `0110` ($6_{10}$)


In [2]:
BLOCK_SIZE = 4
KEY_SIZE = 4
KEY_SPACE_SIZE = 2 ** KEY_SIZE

PLAINTEXT = 10   # Binary: 1010
SECRET_KEY = 6   # Binary: 0110

print("=== Experiment Parameters ===")
print(f"Block Size (n)        : {BLOCK_SIZE} bits")
print(f"Key Space Size (N=2^n): {KEY_SPACE_SIZE} possible keys")
print_bits("Known Plaintext (P)", PLAINTEXT, BLOCK_SIZE)
print_bits("Secret Key (K)", SECRET_KEY, KEY_SIZE)

=== Experiment Parameters ===
Block Size (n)        : 4 bits
Key Space Size (N=2^n): 16 possible keys
Known Plaintext (P)   : 1010 (decimal: 10)
Secret Key (K)        : 0110 (decimal: 6)


## 3. Classical Encryption & Decryption Implementation

In a toy XOR cipher:
- **Encryption**: $C = P \oplus K$
- **Decryption**: $P = C \oplus K$

Because XOR is an involution ($x \oplus y \oplus y = x$), applying the key a second time restores the original plaintext:
$$(P \oplus K) \oplus K = P \oplus (K \oplus K) = P \oplus 0 = P$$


In [3]:
def xor_encrypt(plaintext: int, key: int) -> int:
    """Encrypts a plaintext block with a key using bitwise XOR."""
    return plaintext ^ key

def xor_decrypt(ciphertext: int, key: int) -> int:
    """Decrypts a ciphertext block with a key using bitwise XOR."""
    return ciphertext ^ key

# 1. Execute Encryption
CIPHERTEXT = xor_encrypt(PLAINTEXT, SECRET_KEY)

print("--- Classical Encryption ---")
print_bits("Plaintext (P)", PLAINTEXT, BLOCK_SIZE)
print_bits("Secret Key (K)", SECRET_KEY, KEY_SIZE)
print("-" * 40 + " (P XOR K)")
print_bits("Ciphertext (C)", CIPHERTEXT, BLOCK_SIZE)

# 2. Execute Decryption
DECRYPTED_PLAINTEXT = xor_decrypt(CIPHERTEXT, SECRET_KEY)

print("\n--- Classical Decryption ---")
print_bits("Ciphertext (C)", CIPHERTEXT, BLOCK_SIZE)
print_bits("Secret Key (K)", SECRET_KEY, KEY_SIZE)
print("-" * 40 + " (C XOR K)")
print_bits("Decrypted Plaintext", DECRYPTED_PLAINTEXT, BLOCK_SIZE)

# Verification assertion
assert DECRYPTED_PLAINTEXT == PLAINTEXT, "Error: Decrypted text does not match original plaintext!"
print("\n[VERIFIED] Symmetric Decryption successful: Decrypt(Encrypt(P, K), K) == P")

--- Classical Encryption ---
Plaintext (P)         : 1010 (decimal: 10)
Secret Key (K)        : 0110 (decimal: 6)
---------------------------------------- (P XOR K)
Ciphertext (C)        : 1100 (decimal: 12)

--- Classical Decryption ---
Ciphertext (C)        : 1100 (decimal: 12)
Secret Key (K)        : 0110 (decimal: 6)
---------------------------------------- (C XOR K)
Decrypted Plaintext   : 1010 (decimal: 10)

[VERIFIED] Symmetric Decryption successful: Decrypt(Encrypt(P, K), K) == P


## 4. Known-Plaintext Attack (KPA) Threat Model

In a **Known-Plaintext Attack (KPA)**:
- **Adversary knows**:
  1. The cipher specification ($C = P \oplus K$).
  2. A known plaintext block: $P = 1010_2$ ($10_{10}$).
  3. The intercepted target ciphertext block: $C = 1100_2$ ($12_{10}$).
- **Adversary does NOT know**:
  - The secret key $K$.
- **Adversary's Goal**:
  - Recover $K$ from the key space $\mathcal{K} = \{0, 1\}^4$.


### Attack Method 1: Exhaustive Search (Classical Brute-Force)

For generic black-box ciphers (such as AES or complex block ciphers with non-linear S-boxes), no simple algebraic equation exists to invert the key directly.
The adversary must perform an **exhaustive search** testing candidate keys:

$$\text{For each } k \in \{0, \dots, N-1\}: \quad \text{Evaluate } C_{\text{test}} = P \oplus k \stackrel{?}{=} C$$

- **Search space**: $N = 2^4 = 16$
- **Worst-case query complexity**: $N = 16$ checks
- **Average-case query complexity**: $N / 2 = 8$ checks


In [4]:
def classical_brute_force_kpa(known_pt: int, target_ct: int, key_space_size: int = 16):
    """
    Performs a classical exhaustive key search (brute-force) under a Known-Plaintext Attack.
    Tests each candidate key in the keyspace against the known (P, C) pair.
    """
    print(f"[*] Starting Classical Exhaustive Search over N = {key_space_size} keys...\n")
    print(f"{'Candidate (k)':<15} | {'Candidate (Bin)':<16} | {'Computed C = P^k':<18} | {'Status'}")
    print("=" * 68)
    
    recovered_keys = []
    total_evaluations = 0
    first_match_queries = None
    
    for k in range(key_space_size):
        total_evaluations += 1
        test_ct = xor_encrypt(known_pt, k)
        
        if test_ct == target_ct:
            status = "--> MATCH FOUND (Key Identified!)"
            recovered_keys.append(k)
            if first_match_queries is None:
                first_match_queries = total_evaluations
        else:
            status = "Mismatch"
            
        print(f"{k:<15} | {to_bin(k):<16} | {to_bin(test_ct):<18} | {status}")
        
    print("=" * 68)
    print(f"\nSearch Summary:")
    print(f"- Total candidate keys evaluated (Worst Case): {total_evaluations} / {key_space_size}")
    print(f"- Queries needed to find first match (Average Case test): {first_match_queries}")
    print(f"- Recovered Key(s): {[to_bin(k) for k in recovered_keys]} (Decimal: {recovered_keys})")
    
    return recovered_keys, total_evaluations

recovered_keys, queries = classical_brute_force_kpa(PLAINTEXT, CIPHERTEXT, KEY_SPACE_SIZE)

assert SECRET_KEY in recovered_keys, "Brute-force failed to identify secret key!"
print(f"\n[SUCCESS] Secret key {to_bin(SECRET_KEY)} successfully recovered using Classical Brute Force.")

[*] Starting Classical Exhaustive Search over N = 16 keys...

Candidate (k)   | Candidate (Bin)  | Computed C = P^k   | Status
0               | 0000             | 1010               | Mismatch
1               | 0001             | 1011               | Mismatch
2               | 0010             | 1000               | Mismatch
3               | 0011             | 1001               | Mismatch
4               | 0100             | 1110               | Mismatch
5               | 0101             | 1111               | Mismatch
6               | 0110             | 1100               | --> MATCH FOUND (Key Identified!)
7               | 0111             | 1101               | Mismatch
8               | 1000             | 0010               | Mismatch
9               | 1001             | 0011               | Mismatch
10              | 1010             | 0000               | Mismatch
11              | 1011             | 0001               | Mismatch
12              | 1100             | 0110   

### Attack Method 2: Direct Algebraic Shortcut (XOR-Specific)

Because the XOR operation is linearly self-invertible:
$$C = P \oplus K \iff K = P \oplus C$$

For this toy cipher, an adversary can recover $K$ in a single step ($\mathcal{O}(1)$ query complexity).
*Note: Real-world ciphers prevent this via non-linear confusion (S-boxes) and multi-round diffusion layers.*


In [5]:
direct_key = PLAINTEXT ^ CIPHERTEXT

print("--- Direct Algebraic Key Recovery (XOR property) ---")
print_bits("Known Plaintext (P)", PLAINTEXT, BLOCK_SIZE)
print_bits("Target Ciphertext (C)", CIPHERTEXT, BLOCK_SIZE)
print("-" * 40 + " (P XOR C)")
print_bits("Directly Recovered (K)", direct_key, KEY_SIZE)

assert direct_key == SECRET_KEY, "Algebraic recovery mismatch!"
print(f"\n[VERIFIED] Direct algebraic inversion confirms key = {to_bin(SECRET_KEY)}")

--- Direct Algebraic Key Recovery (XOR property) ---
Known Plaintext (P)   : 1010 (decimal: 10)
Target Ciphertext (C) : 1100 (decimal: 12)
---------------------------------------- (P XOR C)
Directly Recovered (K): 0110 (decimal: 6)

[VERIFIED] Direct algebraic inversion confirms key = 0110


## 5. Algorithmic Complexity: Classical vs. Quantum Search

| Characteristic | Classical Exhaustive Search (This Notebook) | Grover's Quantum Search (Notebook 02) |
| :--- | :--- | :--- |
| **Search Space ($N$)** | $N = 2^n = 16$ keys | $N = 2^n = 16$ quantum basis states |
| **Query Complexity (Worst Case)** | $N = 16$ oracle evaluations | $\approx \frac{\pi}{4}\sqrt{N} \approx 3$ Grover iterations |
| **Asymptotic Complexity** | $\mathcal{O}(N)$ linear time | $\mathcal{O}(\sqrt{N})$ quadratic speedup |
| **Mechanism** | Sequential or parallel classical testing | Superposition ($H^{\otimes n}$) + Phase Inversion + Diffusion |
| **Hardware** | Standard Classical CPU | Superconducting QPU / Quantum Simulator (`AerSimulator`) |

---

### Next Step: Quantum Implementation
Now that the classical baseline and encryption/decryption mechanics are verified, open:
👉 **`notebooks/02_Experiment_Tiny_XOR_Grover_Key_Recovery.ipynb`**
to explore how Grover's Quantum Algorithm solves the exact same key-recovery problem on quantum circuits and physical quantum hardware (`ibm_marrakesh`).
